# PartaGPU — Entraînement distribué multi-GPU

Ce notebook montre comment utiliser PartaGPU pour exploiter les GPU de plusieurs machines d'une salle de cours.

## Prérequis

1. **L'application PartaGPU** doit tourner sur cette machine (et sur les machines qui partagent)
2. **Une salle** doit être configurée (même code d'accès sur toutes les machines)
3. **Le partage** doit être activé sur les machines distantes
4. Le package Python : `pip install partagpu`

## 1. Installation du package

In [ ]:
!pip install partagpu

## 2. Vérifier la connexion avec PartaGPU

Le package communique avec l'application PartaGPU via une API HTTP locale (localhost:7654).

In [ ]:
import requests

try:
    r = requests.get("http://127.0.0.1:7654/api/status", timeout=2)
    print("PartaGPU est en marche !")
    print(f"Statut : {r.json()}")
except requests.ConnectionError:
    print("PartaGPU n'est pas lance. Ouvrez l'application d'abord.")

## 3. Découvrir les machines du réseau

On peut voir toutes les machines qui font tourner PartaGPU sur le réseau local.

In [ ]:
from partagpu.discover import get_peers

peers = get_peers()
print(f"{len(peers)} machine(s) detectee(s) sur le reseau :\n")

for p in peers:
    status = "Verifie" if p.verified else "Non verifie"
    partage = "Actif" if p.sharing_enabled else "Inactif"
    print(f"  {p.display_name} ({p.hostname})")
    print(f"    IP: {p.ip} | Auth: {status} | Partage: {partage}")
    print(f"    CPU: {p.cpu_limit}% | RAM: {p.ram_limit} Mo | GPU: {p.gpu_limit}%")
    print()

## 4. Découvrir les GPU disponibles

C'est la fonction principale : elle retourne la liste des GPU utilisables (le vôtre + ceux des pairs vérifiés qui partagent).

In [ ]:
import partagpu

gpus = partagpu.discover()

print(f"{len(gpus)} GPU disponible(s) pour l'entrainement :\n")
for g in gpus:
    print(f"  {g}")

if len(gpus) == 0:
    print("Aucun GPU detecte.")
    print("Verifiez que :")
    print("  - Vous avez un GPU NVIDIA avec les drivers installes")
    print("  - Les machines distantes ont active le partage")
    print("  - Vous etes dans la meme salle")

## 5. Entraînement distribué avec PyTorch

Exemple simple : entraîner un réseau de neurones sur tous les GPU disponibles.

Le context manager `distribute()` configure automatiquement PyTorch DDP (Distributed Data Parallel).

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Verifier que CUDA est disponible
print(f"CUDA disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU local : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} Go")

In [ ]:
# Exemple : reseau simple pour classifier des donnees synthetiques

# Donnees synthetiques
X = torch.randn(1000, 784)
y = torch.randint(0, 10, (1000,))
dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

# Modele
model = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 10),
)

# Entrainement sur le GPU local
if torch.cuda.is_available():
    model = model.cuda()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    total_loss = 0
    for batch_X, batch_y in loader:
        if torch.cuda.is_available():
            batch_X, batch_y = batch_X.cuda(), batch_y.cuda()
        
        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/5 — Loss: {total_loss/len(loader):.4f}")

print("\nEntrainement termine !")

## 6. Utilisation de l'API HTTP directement

Si vous préférez intégrer PartaGPU dans un outil qui n'est pas Python, vous pouvez utiliser l'API HTTP directement.

In [ ]:
import requests
import json

# Liste des pairs
peers = requests.get("http://127.0.0.1:7654/api/peers").json()
print("=== Pairs ===")
print(json.dumps(peers, indent=2, ensure_ascii=False))

# GPU disponibles
gpus = requests.get("http://127.0.0.1:7654/api/gpu").json()
print("\n=== GPU ===")
print(json.dumps(gpus, indent=2, ensure_ascii=False))

# Statut local
status = requests.get("http://127.0.0.1:7654/api/status").json()
print("\n=== Statut ===")
print(json.dumps(status, indent=2, ensure_ascii=False))